# 02. Exploratory Data Analysis (EDA) — EMIPredict AI

This notebook analyzes `data/processed/cleaned_dataset.csv` to generate visualizations for loan underwriters. Charts are exported to `docs/eda_assets/`.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
sns.set_theme(style="darkgrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

os.makedirs('../docs/eda_assets', exist_ok=True)
os.makedirs('docs/eda_assets', exist_ok=True)
assets_dir = '../docs/eda_assets' if os.path.exists('../docs') else 'docs/eda_assets'

# Load cleaned data
data_path = '../data/processed/cleaned_dataset.csv' if os.path.exists('../data') else 'data/processed/cleaned_dataset.csv'
df = pd.read_csv(data_path)
print(f"Loaded dataset with {len(df):,} records for EDA.")

Loaded dataset with 404,800 records for EDA.


In [2]:
# Chart 1: Eligibility Distribution Overall & by EMI Scenario
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
ax1 = sns.countplot(data=df, x='emi_eligibility', palette='viridis')
plt.title('Overall Eligibility Class Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Eligibility Class')
plt.ylabel('Applicant Count')

plt.subplot(1, 2, 2)
ax2 = sns.countplot(data=df, x='emi_scenario', hue='emi_eligibility', palette='viridis')
plt.title('Eligibility Breakdown by EMI Scenario', fontsize=12, fontweight='bold')
plt.xlabel('EMI Scenario')
plt.ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()

chart1_path = os.path.join(assets_dir, 'eligibility_distribution.png')
plt.savefig(chart1_path, dpi=300)
plt.close()
print(f"Saved Chart 1 to {chart1_path}")

/tmp/ipykernel_4607/2810869258.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax1 = sns.countplot(data=df, x='emi_eligibility', palette='viridis')


Saved Chart 1 to ../docs/eda_assets/eligibility_distribution.png


### Underwriter Interpretation — Chart 1 (Overall & Scenario Distribution)

- **Class Balance**: Shows the distribution across `Eligible`, `High_Risk`, and `Not_Eligible` categories.
- **Scenario Variance**: Identifies product categories with higher risk concentration (e.g. Unsecured Personal Loans vs. Collateralized Vehicle Loans).

In [3]:
# Chart 2: Correlation Heatmap of Financial Variables
num_cols = ['monthly_salary', 'credit_score', 'bank_balance', 'monthly_rent', 'current_emi_amount', 'requested_amount', 'max_monthly_emi']
plt.figure(figsize=(9, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Financial Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()

chart2_path = os.path.join(assets_dir, 'correlation_heatmap.png')
plt.savefig(chart2_path, dpi=300)
plt.close()
print(f"Saved Chart 2 to {chart2_path}")

Saved Chart 2 to ../docs/eda_assets/correlation_heatmap.png


### Underwriter Interpretation — Chart 2 (Feature Correlation Matrix)

- **Income vs. EMI Limit**: Evaluates line-of-credit scaling relative to gross monthly salary.
- **Credit Score & Liquidity**: Measures co-linearity between credit rating, liquid savings, and existing monthly obligations.

In [4]:
# Chart 3: Approval Rates by Age Bracket, Employment Sector & Education
df['age_bracket'] = pd.cut(df['age'], bins=[20, 30, 40, 50, 65], labels=['21-30', '31-40', '41-50', '51-60'])
df['is_approved'] = (df['emi_eligibility'] == 'Eligible').astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.barplot(data=df, x='age_bracket', y='is_approved', ax=axes[0], palette='Blues_d')
axes[0].set_title('Approval Rate by Age Bracket')
axes[0].set_ylabel('Approval Rate')

sns.barplot(data=df, x='employment_type', y='is_approved', ax=axes[1], palette='Greens_d')
axes[1].set_title('Approval Rate by Employment Type')
axes[1].set_ylabel('')

sns.barplot(data=df, x='education', y='is_approved', ax=axes[2], palette='Purples_d')
axes[2].set_title('Approval Rate by Education')
axes[2].set_ylabel('')
plt.tight_layout()

chart3_path = os.path.join(assets_dir, 'demographic_approval_rates.png')
plt.savefig(chart3_path, dpi=300)
plt.close()
print(f"Saved Chart 3 to {chart3_path}")

/tmp/ipykernel_4607/2153432297.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='age_bracket', y='is_approved', ax=axes[0], palette='Blues_d')


/tmp/ipykernel_4607/2153432297.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='employment_type', y='is_approved', ax=axes[1], palette='Greens_d')


/tmp/ipykernel_4607/2153432297.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='education', y='is_approved', ax=axes[2], palette='Purples_d')


Saved Chart 3 to ../docs/eda_assets/demographic_approval_rates.png


### Underwriter Interpretation — Chart 3 (Demographic Approval Trends)

- **Age Stability**: Mid-career applicants (31-50) display higher credit stability.
- **Employment Sector**: Public sector / Government employees show reliable low-risk baselines.

In [5]:
# Chart 4: Max Monthly EMI Boxplot by Scenario
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='emi_scenario', y='max_monthly_emi', palette='Set2')
plt.title('Max Recommended Monthly EMI Distribution by Scenario (INR)', fontsize=12, fontweight='bold')
plt.xlabel('Loan Scenario')
plt.ylabel('Max EMI (INR)')
plt.xticks(rotation=15)
plt.tight_layout()

chart4_path = os.path.join(assets_dir, 'max_emi_boxplots.png')
plt.savefig(chart4_path, dpi=300)
plt.close()
print(f"Saved Chart 4 to {chart4_path}")

/tmp/ipykernel_4607/3593805933.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='emi_scenario', y='max_monthly_emi', palette='Set2')


Saved Chart 4 to ../docs/eda_assets/max_emi_boxplots.png


### Underwriter Interpretation — Chart 4 (Max EMI Boxplots)

- **Recommended Ceiling**: Outlines median, upper, and lower quartile monthly repayment caps across loan products.

In [6]:
# Export Real EDA Summary to public/eda-summary.json (D5)
import json

eda_json_path = '../public/eda-summary.json' if os.path.exists('../public') else 'public/eda-summary.json'

# 1. eligibilityByScenario
scenario_counts = df.groupby(['emi_scenario', 'emi_eligibility']).size().unstack(fill_value=0)
eligibilityByScenario = []
for sc in scenario_counts.index:
    total = scenario_counts.loc[sc].sum()
    eligibilityByScenario.append({
        'scenario': sc,
        'Eligible': round(float(scenario_counts.loc[sc].get('Eligible', 0) / total * 100), 1),
        'High_Risk': round(float(scenario_counts.loc[sc].get('High_Risk', 0) / total * 100), 1),
        'Not_Eligible': round(float(scenario_counts.loc[sc].get('Not_Eligible', 0) / total * 100), 1)
    })

# 2. eligibilityByAgeBracket
age_bins = [25, 30, 40, 50, 60]
age_labels = ['25-30', '31-40', '41-50', '51-60']
df['age_bracket_temp'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, include_lowest=True)
age_elig = df.groupby('age_bracket_temp')['emi_eligibility'].apply(lambda s: (s == 'Eligible').mean() * 100)
eligibilityByAgeBracket = [{'ageBracket': str(k), 'approvalRate': round(float(v), 1)} for k, v in age_elig.items()]

# 3. eligibilityByEmploymentType
emp_elig = df.groupby('employment_type')['emi_eligibility'].apply(lambda s: (s == 'Eligible').mean() * 100)
eligibilityByEmploymentType = [{'type': str(k), 'approvalRate': round(float(v), 1)} for k, v in emp_elig.items()]

# 4. emiDistributionByScenario
emiDistributionByScenario = []
for sc, group in df.groupby('emi_scenario'):
    vals = group['max_monthly_emi'].dropna()
    emiDistributionByScenario.append({
        'scenario': sc,
        'min': float(vals.min()),
        'q1': float(vals.quantile(0.25)),
        'median': float(vals.median()),
        'q3': float(vals.quantile(0.75)),
        'max': float(vals.max())
    })

eda_payload = {
    'generated': True,
    'generated_at': pd.Timestamp.now().isoformat(),
    'eligibilityByScenario': eligibilityByScenario,
    'eligibilityByAgeBracket': eligibilityByAgeBracket,
    'eligibilityByEmploymentType': eligibilityByEmploymentType,
    'emiDistributionByScenario': emiDistributionByScenario
}

with open(eda_json_path, 'w') as f:
    json.dump(eda_payload, f, indent=2)
print(f"✓ Successfully exported honest real EDA dataset metrics to {eda_json_path} (generated: true)")

/tmp/ipykernel_4607/33256499.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_elig = df.groupby('age_bracket_temp')['emi_eligibility'].apply(lambda s: (s == 'Eligible').mean() * 100)


✓ Successfully exported honest real EDA dataset metrics to ../public/eda-summary.json (generated: true)


## Key Business Insights

- [FILL IN AFTER RUNNING — Bullet 1: Summarize top driver of eligibility class selection]
- [FILL IN AFTER RUNNING — Bullet 2: Summarize observed impact of employment type on EMI caps]
- [FILL IN AFTER RUNNING — Bullet 3: Summarize debt-to-income threshold correlations]